In [ ]:
from dotenv import load_dotenv
from pprint import pprint
import os
from openai import OpenAI
import json
import time

load_dotenv()

# helper function to load json files
def load_json(file_path: str) -> dict:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

simple_wiki_data = load_json("simple_wiki_raw_data.json")
normal_wiki_data = load_json("normal_wiki_raw_data.json")



In [ ]:

# ---------- Merging simple and normal page data ----------


# Index both datasets by title (simpler for retrieval)
simple_index = {d['title']: d for d in simple_wiki_data}
normal_index = {d['title']: d for d in normal_wiki_data}


# Merge into a list of dicts
wiki_data = [
    {
        "title": title,
        "simple_wiki": {k: v for k, v in simple_index.get(title, {}).items() if k != "title"},
        "normal_wiki": {k: v for k, v in normal_index.get(title, {}).items() if k != "title"},
    }
    for title in set(simple_index) | set(normal_index) # Union of titles in simple and normal wiki data
]

# store in json file
merged_path = 'merged_wiki_data.json'
with open(merged_path, "w", encoding="utf-8") as f:
        json.dump(wiki_data, f, ensure_ascii=False, indent=2)

pprint(wiki_data[:2], width=200, sort_dicts=False)


[{'title': 'Regression toward the mean',
  'simple_wiki': {'url': 'https://simple.wikipedia.org/wiki/Regression_toward_the_mean',
                  'sections': [{'heading': 'Introduction',
                                'paragraphs': ['Regression toward the mean simply means that, following an extreme random event, the next random event is likely to be less extreme. Regression toward '
                                               'the mean was first described by Francis Galton. He found that offspring of tall parents tended to be shorter. Also, offspring of shorter parents '
                                               'tended to be taller. Galton stated that processes that did not follow regression towards the mean would quickly go out of control.']},
                               {'heading': 'History',
                                'paragraphs': ['In 1886, Galton published a paper called Regression towards mediocrity in hereditary stature. In the paper, he observed that extre

In [ ]:

# ------------ Definition for kids - Generation ------------

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 20000

merged_wiki_data = load_json('merged_wiki_data.json')


pages = []

for page in merged_wiki_data:
    # Pick the available source (by default simple_wiki because usually shorter)
    description = (
        page.get("simple_wiki", {}).get("sections") 
        or page.get("normal_wiki", {}).get("sections") 
        or ""
        )
    description = description[:MAX_INPUT_TOKENS]
    
    try:
            completion = client.chat.completions.create(
			model="meta-llama/Llama-3.1-8B-Instruct:novita",
			messages=[
				{"role": "system", "content": "You explain concepts clearly for children around 10 years old. Use simple words, short sentences, and concrete examples. Avoid technical terms unless they are explained. Do not include introductions, titles, or meta commentary. Output only the explanation."},
				{"role": "user", "content": f"Explain this so a 10-year-old can understand it:\n\ntopic: {page['title']}\ndescription: {description}"}
			],
			max_completion_tokens=500,
            )
            simply_explained_10yo = completion.choices[0].message.content
            
    except Exception as e:
        print(f"Skipping {page['title']} due to API error: {e}")
        simply_explained_10yo = None
        
    pages.append({
        "title": page['title'],
        "technical_definition": page.get('normal_wiki', {}).get('sections'), 
        "simple_definition": page.get('simple_wiki', {}).get('sections'),
        "definition4kids": simply_explained_10yo
    })
    
    time.sleep(0.5) # avoid hammering API


# store as json file
full_wiki_data_path = "full_wiki_data.json"
with open(full_wiki_data_path, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

# quick check
pprint(pages[:2], width=300, sort_dicts=False)


[{'title': 'Regression toward the mean',
  'technical_definition': [{'heading': 'Introduction',
                            'paragraphs': ['In statistics, regression toward the mean (also called regression to the mean, reversion to the mean, and reversion to mediocrity ) is the phenomenon where if one sample of a random variable is extreme, the next sampling of the same random variable is '
                                           'likely to be closer to its mean. Furthermore, when many random variables are sampled and the most extreme results are intentionally picked out, it refers to the fact that (in many cases) a second sampling of these picked-out variables will result in '
                                           '"less extreme" results, closer to the initial mean of all of the variables.',
                                           'Mathematically, the strength of this "regression" effect is dependent on whether or not all of the random variables are drawn from the same dist

# Display

In [ ]:
from IPython.display import display, Markdown

SPLIT = "\n\n" + "-"*66 + "\n"

def format_sections(title: str, sections: list) -> str:
    """Format a list of sections with a heading and paragraphs"""
    if not sections:
        return ""
    content = [SPLIT + f"## {title}:\n"]  # SPLIT at the start of this section
    for section in sections:
        content.append(f"### {section['heading']}\n")
        content.extend(para + "\n" for para in section.get("paragraphs", []))
    return "\n".join(content)

def pretty_print_page_global(page: dict, definition_type: str | None = None) -> None:
    """
    Print a single page.
    
    definition_type: Optional[str] - one of "technical", "simple", "kids".
                     If None, prints all definitions.
    """
    md = [f"{SPLIT*2}# {page.get('title', 'Untitled')}\n"]

    sections_map = {
        "technical": ("Technical Definition", page.get("technical_definition", [])),
        "simple": ("Simple Definition", page.get("simple_definition", [])),
        "kids": ("Definition Simplified for 10yo", page.get("definition4kids"))
    }

    if definition_type:
        # Only print the selected definition
        if definition_type == "kids" and sections_map["kids"][1]:
            md.append(SPLIT + f"## {sections_map['kids'][0]}:\n{sections_map['kids'][1]}\n")
        elif definition_type in ["technical", "simple"]:
            md.append(format_sections(*sections_map[definition_type]))
        else:
            md.append(f"{SPLIT}**No content found for '{definition_type}'**")
    else:
        # Print all available definitions
        for key, value in sections_map.items():
            if key == "kids" and value[1]:
                md.append(SPLIT + f"## {value[0]}:\n{value[1]}\n")
            elif key in ["technical", "simple"]:
                md.append(format_sections(*value))

    display(Markdown("\n".join(md)))

def pretty_print_all(pages: list, definition_type: str | None = None) -> None:
    """Print every pages"""
    for page in pages:
        pretty_print_page_global(page, definition_type)
    display(Markdown(SPLIT*2))


pretty_print_page_global(pages[2])

pretty_print_all(pages[:2], definition_type='kids')



------------------------------------------------------------------


------------------------------------------------------------------
# Software



------------------------------------------------------------------
## Technical Definition:

### Introduction

Software consists of computer programs that instruct the execution of a computer. Software also includes design documents and specifications.

The history of software is closely tied to the development of digital computers in the mid-20th century. Early programs were written in the machine language specific to the hardware. The introduction of high-level programming languages in 1958 allowed for more human-readable instructions, making software development easier and more portable across different computer architectures. Software in a programming language is run through a compiler or interpreter to execute on the architecture's hardware. Over time, software has become complex, owing to developments in networking, operating systems, and databases.

Software can generally be categorized into two main types:

1) operating systems, which manage hardware resources and provide services for applications
2) application software, which performs specific tasks for users

operating systems, which manage hardware resources and provide services for applications

application software, which performs specific tasks for users

The rise of cloud computing has introduced the new software delivery model Software as a Service (SaaS). In SaaS, applications are hosted by a provider and accessed over the Internet.

The process of developing software involves several stages. The stages include software design, programming, testing, release, and maintenance. Software quality assurance and security are critical aspects of software development, as bugs and security vulnerabilities can lead to system failures and security breaches. Additionally, legal issues such as software licenses and intellectual property rights play a significant role in the distribution of software products.

### History

The first use of the word software to describe computer programs is credited to mathematician John Wilder Tukey in 1958. The first programmable computers, which appeared at the end of the 1940s, were programmed in machine language. Machine language is difficult to debug and not portable across different computers. Initially, hardware resources were more expensive than human resources. As programs became complex, programmer productivity became the bottleneck. The introduction of high-level programming languages in 1958 hid the details of the hardware and expressed the underlying algorithms into the code. Early languages include Fortran, Lisp, and COBOL.

### Types

There are two main types of software:

- Operating systems are "the layer of software that manages a computer's resources for its users and their applications ". There are three main purposes that an operating system fulfills: Allocating resources between different applications, deciding when they will receive central processing unit (CPU) time or space in memory. Providing an interface that abstracts the details of accessing hardware details (like physical memory) to make things easier for programmers. Offering common services, such as an interface for accessing network and disk devices. This enables an application to be run on different hardware without needing to be rewritten.
- Application software runs on top of the operating system and uses the computer's resources to perform a task. There are many different types of application software because the range of tasks that can be performed with modern computers is so large. Applications account for most software and require the environment provided by an operating system, and often other applications, in order to function.

Operating systems are "the layer of software that manages a computer's resources for its users and their applications ". There are three main purposes that an operating system fulfills: Allocating resources between different applications, deciding when they will receive central processing unit (CPU) time or space in memory. Providing an interface that abstracts the details of accessing hardware details (like physical memory) to make things easier for programmers. Offering common services, such as an interface for accessing network and disk devices. This enables an application to be run on different hardware without needing to be rewritten.

- Allocating resources between different applications, deciding when they will receive central processing unit (CPU) time or space in memory.
- Providing an interface that abstracts the details of accessing hardware details (like physical memory) to make things easier for programmers.
- Offering common services, such as an interface for accessing network and disk devices. This enables an application to be run on different hardware without needing to be rewritten.

Allocating resources between different applications, deciding when they will receive central processing unit (CPU) time or space in memory.

Providing an interface that abstracts the details of accessing hardware details (like physical memory) to make things easier for programmers.

Offering common services, such as an interface for accessing network and disk devices. This enables an application to be run on different hardware without needing to be rewritten.

Application software runs on top of the operating system and uses the computer's resources to perform a task. There are many different types of application software because the range of tasks that can be performed with modern computers is so large. Applications account for most software and require the environment provided by an operating system, and often other applications, in order to function.

Software can also be categorized by how it is deployed. Traditional applications are purchased with a perpetual license for a specific version of the software, downloaded, and run on hardware belonging to the purchaser. The rise of the Internet and cloud computing enabled a new model, software as a service (SaaS), in which the provider hosts the software (usually built on top of rented infrastructure or platforms ) and provides the use of the software to customers, often in exchange for a subscription fee. By 2023, SaaS products—which are usually delivered via a web application —had become the primary method that companies deliver applications.

### Software development and maintenance

Software companies aim to deliver a high-quality product on time and under budget. A challenge is that software development effort estimation is often inaccurate. Software development begins by conceiving the project, evaluating its feasibility, analyzing the business requirements, and making a software design. Most software projects speed up their development by reusing or incorporating existing software, either in the form of commercial off-the-shelf (COTS) or open-source software. Software quality assurance is typically a combination of manual code review by other engineers and automated software testing. Due to time constraints, testing cannot cover all aspects of the software's intended functionality, so developers often focus on the most critical functionality. Formal methods are used in some safety-critical systems to prove the correctness of code, while user acceptance testing helps to ensure that the product meets customer expectations. There are a variety of software development methodologies, which vary from completing all steps in order to concurrent and iterative models. Software development is driven by requirements taken from prospective users, as opposed to maintenance, which is driven by events such as a change request.

Frequently, software is released in an incomplete state when the development team runs out of time or funding. Despite testing and quality assurance, virtually all software contains bugs where the system does not work as intended. Post-release software maintenance is necessary to remediate these bugs when they are found and keep the software working as the environment changes over time. New features are often added after the release. Over time, the level of maintenance becomes increasingly restricted before being cut off entirely when the product is withdrawn from the market. As software ages, it becomes known as legacy software and can remain in use for decades, even if there is no one left who knows how to fix it. Over the lifetime of the product, software maintenance is estimated to comprise 75 percent or more of the total development cost.

Completing a software project involves various forms of expertise, not just in software programmers but also testing, documentation writing, project management, graphic design, user experience, user support, marketing, and fundraising.

### Quality and security

Software quality is defined as meeting the stated requirements as well as customer expectations. Quality is an overarching term that can refer to a code's correct and efficient behavior, its reusability and portability, or the ease of modification. It is usually more cost-effective to build quality into the product from the beginning rather than try to add it later in the development process. Higher quality code will reduce lifetime cost to both suppliers and customers as it is more reliable and easier to maintain. Software failures in safety-critical systems may result in serious harm, including injury or death. By some estimates, the cost of poor quality software can be as high as 20 to 40 percent of sales. Despite developers' goal of delivering a product that works entirely as intended, virtually all software contains bugs.

The rise of the Internet also greatly increased the need for computer security as it enabled malicious actors to conduct cyberattacks remotely. If a bug creates a security risk, it is called a vulnerability. Software patches are often released to fix identified vulnerabilities, but those that remain unknown ( zero days ) as well as those that have not been patched are still liable for exploitation. Vulnerabilities vary in their ability to be exploited by malicious actors, and the actual risk is dependent on the nature of the vulnerability as well as the value of the surrounding system. Although some vulnerabilities can only be used for denial of service attacks that compromise a system's availability, others allow the attacker to inject and run their own code (called malware ), without the user being aware of it. To thwart cyberattacks, all software in the system must be designed to withstand and recover from external attack. Despite efforts to ensure security, a significant fraction of computers are infected with malware.

### Encoding and execution

### Programming languages

Programming languages are the format in which software is written. Since the 1950s, thousands of different programming languages have been invented; some have been in use for decades, while others have fallen into disuse. Some definitions classify machine code —the exact instructions directly implemented by the hardware—and assembly language —a more human-readable alternative to machine code whose statements can be translated one-to-one into machine code—as programming languages. Programs written in the high-level programming languages used to create software share a few main characteristics: knowledge of machine code is not necessary to write them, they can be ported to other computer systems, and they are more concise and human-readable than machine code. They must be both human-readable and capable of being translated into unambiguous instructions for computer hardware.

### Compilation,  interpretation, and execution

The invention of high-level programming languages was simultaneous with the compilers needed to translate them automatically into machine code. Most programs do not contain all the resources needed to run them and rely on external libraries. Part of the compiler's function is to link these files in such a way that the program can be executed by the hardware. Once compiled, the program can be saved as an object file and the loader (part of the operating system) can take this saved file and execute it as a process on the computer hardware. Some programming languages use an interpreter instead of a compiler. An interpreter converts the program into machine code at run time, which makes them 10 to 100 times slower than compiled programming languages.

### Legal issues

### Liability

Software is often released with the knowledge that it is incomplete or contains bugs. Purchasers knowingly buy it in this state, which has led to a legal regime where liability for software products is significantly curtailed compared to other products.

### Licenses

Since the mid-1970s, software and its source code have been protected by copyright law that vests the owner with the exclusive right to copy the code. The underlying ideas or algorithms are not protected by copyright law, but are sometimes treated as a trade secret and concealed by such methods as non-disclosure agreements. A software copyright is often owned by the person or company that financed or made the software (depending on their contracts with employees or contractors who helped to write it). Some software is in the public domain and has no restrictions on who can use it, copy or share it, or modify it; a notable example is software written by the United States Government. Free and open-source software also allow free use, sharing, and modification, perhaps with a few specified conditions. The use of some software is governed by an agreement ( software license ) written by the copyright holder and imposed on the user. Proprietary software is usually sold under a restrictive license that limits its use and sharing. Some free software licenses require that modified versions must be released under the same license, which prevents the software from being sold or distributed under proprietary restrictions.

### Patents

Patents give an inventor an exclusive, time-limited license for a novel product or process. Ideas about what software could accomplish are not protected by law and concrete implementations are instead covered by copyright law. In some countries, a requirement for the claimed invention to have an effect on the physical world may also be part of the requirements for a software patent to be held valid. Software patents have been historically controversial. Before the 1998 case State Street Bank & Trust Co. v. Signature Financial Group, Inc., software patents were generally not recognized in the United States. In that case, the Supreme Court decided that business processes could be patented. Patent applications are complex and costly, and lawsuits involving patents can drive up the cost of products. Unlike copyrights, patents generally only apply in the jurisdiction where they were issued.

### Impact

Engineer Capers Jones writes that "computers and software are making profound changes to every aspect of human life: education, work, warfare, entertainment, medicine, law, and everything else". It has become ubiquitous in everyday life in developed countries. In many cases, software augments the functionality of existing technologies such as household appliances and elevators. Software also spawned entirely new technologies such as the Internet, video games, mobile phones, and GPS. New methods of communication, including email, forums, blogs, microblogging, wikis, and social media, were enabled by the Internet. Massive amounts of knowledge exceeding any paper-based library are now available with a quick web search. Most creative professionals have switched to software-based tools such as computer-aided design, 3D modeling, digital image editing, and computer animation. Almost every complex device is controlled by software.



------------------------------------------------------------------
## Simple Definition:

### Introduction

Computer software, also called software, is a set of instructions and documentation that tells a computer what to do or how to perform a task. Software includes all different programs on a computer, such as applications and the operating system. Applications are programs that are designed to perform a specific operation, such as a game or a word processor. The operating system (e.g., macOS, Microsoft Windows, Android and various Linux distributions) is a type of software that is used as a platform for running the applications, and controls all user interface tools including display and the keyboard.

The word software was first used in the late 1960s to emphasize on its difference from computer hardware, which can be physically observed by the user. Software is a set of instructions that the computer follows. Before compact discs (CDs) or development of the Internet age, software was used on various computer data storage media tools like paper punch cards, magnetic discs or magnetic tapes.

The word firmware is sometimes used to describe a style of software that is made specifically for a particular type of computer or an electronic device and is usually stored on a Flash memory or ROM chip in the computer. Firmware usually refers to a piece of software that directly controls a piece of hardware. The firmware for a CD drive or the firmware for a modem are examples of firmware implementation.

Today, software has become an important part of our lives. Software is used everywhere. Software engineers are responsible for producing fault-free software, which has become an essential part of our daily lives. Changeability and conformity are two of the main properties of software design. There are also different processing models for designing software including Build and Fix, Waterfall and Agile software processing design methods.

### Types of Software

The different types of software can be put into categories based on common function, type, or field of use. There are three broad classifications:

1) Application software are the computer programs for performing user tasks, such as word processing and web browsers.
2) System software is used to start and run computer systems and networks. This includes operating systems.
3) Computer programming tools (also known as development software ) are used to create application and system software. This is done by translating and combining computer program source code and libraries into executable RAMs. These include compiler and linker.



------------------------------------------------------------------
## Definition Simplified for 10yo:
So, you know how you can tell your friends what to do or how to do something? Like, "Go get a snack" or "Write your name on this paper." Software is like a set of instructions that a computer follows.

Imagine you have a toy robot that can draw pictures. You can tell the robot what to draw by giving it a set of instructions, like "Draw a cat" or "Draw a house." The robot will follow those instructions and draw the picture. That's kind of like how software works.

Software is made up of different programs that help a computer do things. There are games, like Minecraft or Roblox, that you can play on a computer. There are also programs that help you write papers or send emails. And there's the operating system, which is like the brain of the computer that makes everything work.

A long time ago, people had to use special cards or tapes to tell computers what to do. But now, we can use CDs or the internet to store and use software.

There's also something called firmware, which is like a special kind of software that helps a computer's parts work together. It's like the instructions that the robot's drawing arm follows to make sure it draws the right picture.

Today, software is a big part of our lives. We use it to play games, talk to our friends, and even do our homework. And there are special people called software engineers who make sure the software works properly.

Software can be grouped into three main types:

1. Application software: This is like the games or word processors that you use to do things.
2. System software: This is like the operating system that helps the computer run smoothly.
3. Development software: This is like the tools that software engineers use to make new software.

So, that's what software is! It's like a set of instructions that a computer follows to do things.




------------------------------------------------------------------


------------------------------------------------------------------
# Regression toward the mean



------------------------------------------------------------------
## Definition Simplified for 10yo:
When you do something extreme, like running a super-fast 100-meter dash, it's likely that the next time you run the dash, you won't be as fast as you were before. This isn't because you're getting slower, but because you were unusually fast the first time.

A long time ago, a man named Francis Galton studied how tall people were. He found that when two tall parents had a child, the child was usually not as tall as the parents. And when two short parents had a child, the child was usually taller than the parents. This is because the child is not exactly like either parent, but a mix of both.

Imagine you have a big jar of beads, and each bead represents a tiny piece of information about how tall you are. Your parents have their own jars of beads, and when you're born, you get a mix of their beads. Sometimes, you get a lot of beads that make you taller, and sometimes you get a lot of beads that make you shorter. The next time someone measures how tall you are, you're likely to be closer to the average height than you were before.

This happens because the beads that make you taller or shorter are random, and when you're measured again, the random beads are likely to balance out. So, even if you were unusually tall or short before, the next time you're measured, you're likely to be closer to the average.




------------------------------------------------------------------


------------------------------------------------------------------
# Kernel (operating system)



------------------------------------------------------------------
## Definition Simplified for 10yo:
So, you want to know about something called a "kernel". A kernel is like the boss of your computer. It helps everything run smoothly and makes sure that all the different parts of your computer can talk to each other.

Imagine you're at a restaurant and you want to order food. You give your order to a waiter, and the waiter takes it to the kitchen. The kitchen then makes your food and the waiter brings it back to you.

Your computer is like the restaurant, and the kernel is like the waiter. It takes your requests (like orders) and sends them to the "kitchen" (the other parts of the computer) to be handled.

The kernel is also in charge of making sure that everything runs smoothly and that all the different parts of the computer can talk to each other. It's like the kernel is saying, "Okay, the computer is asking for more memory, let me see what I can do."

There are different types of kernels, like monolithic kernels and microkernels. Monolithic kernels are like a big kitchen where everything is made in one place. Microkernels are like a smaller kitchen where some things are made in one place, and others are made in different places.

The kernel also helps protect the computer from bad things that might happen. It's like the kernel is saying, "Don't let that bad guy get in here!" It helps keep the computer safe and running smoothly.

So, that's what a kernel is! It's like the boss of your computer, helping everything run smoothly and keeping the computer safe.




------------------------------------------------------------------


------------------------------------------------------------------
# Software



------------------------------------------------------------------
## Definition Simplified for 10yo:
So, you know how you can tell your friends what to do or how to do something? Like, "Go get a snack" or "Write your name on this paper." Software is like a set of instructions that a computer follows.

Imagine you have a toy robot that can draw pictures. You can tell the robot what to draw by giving it a set of instructions, like "Draw a cat" or "Draw a house." The robot will follow those instructions and draw the picture. That's kind of like how software works.

Software is made up of different programs that help a computer do things. There are games, like Minecraft or Roblox, that you can play on a computer. There are also programs that help you write papers or send emails. And there's the operating system, which is like the brain of the computer that makes everything work.

A long time ago, people had to use special cards or tapes to tell computers what to do. But now, we can use CDs or the internet to store and use software.

There's also something called firmware, which is like a special kind of software that helps a computer's parts work together. It's like the instructions that the robot's drawing arm follows to make sure it draws the right picture.

Today, software is a big part of our lives. We use it to play games, talk to our friends, and even do our homework. And there are special people called software engineers who make sure the software works properly.

Software can be grouped into three main types:

1. Application software: This is like the games or word processors that you use to do things.
2. System software: This is like the operating system that helps the computer run smoothly.
3. Development software: This is like the tools that software engineers use to make new software.

So, that's what software is! It's like a set of instructions that a computer follows to do things.




------------------------------------------------------------------


------------------------------------------------------------------
